In [ ]:
import planetary_computer as pc
from pystac_client import Client
import pandas as pd

catalog = Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=pc.sign_inplace,
)
LAT, LON = 41.878, -93.097
bbox = [LON - 0.01, LAT - 0.01, LON + 0.01, LAT + 0.01]

search = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime="2023-05-01/2023-09-30",
    query={"eo:cloud_cover": {"lt": 20}},
)
items = list(search.items())
print(f"Found {len(items)} cloud-free scenes")

In [ ]:
rows = []
for item in items:
    rows.append({
        "date": item.datetime.strftime("%Y-%m-%d"),
        "cloud_pct": round(item.properties.get("eo:cloud_cover", 0), 1),
        "scene_id": item.id,
    })
inventory = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
print(inventory.to_string())

import os; os.makedirs("data", exist_ok=True)
inventory.to_csv("data/scene_inventory_iowa_2023.csv", index=False)
print("\nSaved inventory")